In [1]:
import pandas as pd
import numpy as np

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [4]:
df = pd.read_csv('appointments.csv')

In [76]:
# Ham verideki tarih, saat ve metin gibi alanları makine öğrenmesi modelinin matematiksel olarak anlamlandırabilmesi için.
df['appointment_date'] = pd.to_datetime(df['appointment_date'])
df['appointment_time'] = pd.to_datetime(df['appointment_time'], format='%H:%M:%S').dt.time

In [32]:
# zaman verilerine model eğitimi için bir uygun forma sokmaya çalışıyorum
df['day_of_week'] = df['appointment_date'].dt.day_name()
df['month'] = df['appointment_date'].dt.month
df['hour'] = pd.to_datetime(df['appointment_time'].astype(str)).dt.hour

In [50]:
# doktor hasta randevu izlenimi
patient_counts = df['patient_id'].value_counts().to_dict()
df['patient_total_appointments'] = df['patient_id'].map(patient_counts)

In [51]:
# doktor counts ve ai bağıntısını çöz.
doctor_counts = df['doctor_id'].value_counts().to_dict()
df['doctor_total_appointments'] = df['doctor_id'].map(doctor_counts)

In [67]:
# Model için kullanılacak sütunların seçilmesi
features_df = df[['reason_for_visit', 'day_of_week', 'month', 'hour', 
                  'patient_total_appointments', 'doctor_total_appointments', 'status']].copy()

In [68]:
features_df = pd.get_dummies(features_df, columns=['reason_for_visit', 'day_of_week'], drop_first=True)

In [69]:
# Tahmin etmek istediğimiz hedef
le = LabelEncoder()
features_df['target'] = le.fit_transform(features_df['status'])
features_df.drop('status', axis=1, inplace=True)

In [64]:
# Girdileri (X) ve tahmin edeceğimiz sonucu (y)
X = features_df.drop('target', axis=1)
y = features_df['target']

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [57]:
# Gradient Boosting modeli
gb_model = GradientBoostingClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=3, 
    random_state=42
)

gb_model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (im

In [58]:

y_pred = gb_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

In [59]:
# Modelin test verisi üzerindeki tahminlerini
print(f"--- Gradient Boosting Sonuçları ---")
print(f"Test Doğruluğu (Accuracy): {acc:.4f}\n")
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

--- Gradient Boosting Sonuçları ---
Test Doğruluğu (Accuracy): 0.3750

Sınıflandırma Raporu:
              precision    recall  f1-score   support

   Cancelled       0.57      0.40      0.47        10
   Completed       0.44      0.44      0.44         9
     No-show       0.38      0.45      0.42        11
   Scheduled       0.18      0.20      0.19        10

    accuracy                           0.38        40
   macro avg       0.40      0.37      0.38        40
weighted avg       0.39      0.38      0.38        40



In [60]:
# test mekanizması
# Test setinden rastgele bir satır seçelim (örneğin ilk satır)
rastgele_index = 0
ornek_girdi = X_test.iloc[[rastgele_index]]
gercek_deger_kodu = y_test.iloc[rastgele_index]

# Model ile tahmin yapalım
tahmin_kodu = gb_model.predict(ornek_girdi)[0]

# Sayısal kodları orijinal metin haline (LabelEncoder ile) geri çevirelim
gercek_durum = le.inverse_transform([gercek_deger_kodu])[0]
tahmin_durumu = le.inverse_transform([tahmin_kodu])[0]

print("--- Test Setinden Örnek Tahmin ---")
print(f"Gerçek Durum: {gercek_durum}")
print(f"Modelin Tahmini: {tahmin_durumu}")

if gercek_durum == tahmin_durumu:
    print("Sonuç: Başarılı Tahmin! 🎉")
else:
    print("Sonuç: Yanlış Tahmin.")

--- Test Setinden Örnek Tahmin ---
Gerçek Durum: Cancelled
Modelin Tahmini: Completed
Sonuç: Yanlış Tahmin.
